# Additional End of week Exercise - week 2

Now use everything you've learned from Week 2 to build a full prototype for the technical question/answerer you built in Week 1 Exercise.

This should include a Gradio UI, streaming, use of the system prompt to add expertise, and the ability to switch between models. Bonus points if you can demonstrate use of a tool!

If you feel bold, see if you can add audio input so you can talk to it, and have it respond with audio. ChatGPT or Claude can help you, or email me if you have questions.

I will publish a full solution here soon - unless someone beats me to it...

There are so many commercial applications for this, from a language tutor, to a company onboarding solution, to a companion AI to a course (like this one!) I can't wait to see your results.

In [5]:
import os
import sys
import json
import httpx
import sqlite3
import gradio as gr

from pathlib import Path
from openai import OpenAI


parent_dir = Path.cwd().parent
sys.path.insert(0, str(parent_dir))
from config import CONFIG

In [2]:
llm_config = CONFIG['llm']
models_config = CONFIG['models']

openai = OpenAI(
    base_url=llm_config['llm_base_url'],
    api_key=llm_config['llm_api_key'],
    http_client=httpx.Client(verify=False, timeout=60)
)

In [4]:


db_path = "library.db"

books = [
    ("The Hobbit", "Central Library"),
    ("Pride and Prejudice", "West Branch"),
    ("1984", "East Branch"),
    ("The Little Prince", "Central Library"),
]

with sqlite3.connect(db_path) as conn:
    conn.execute("""
        CREATE TABLE IF NOT EXISTS books (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            title TEXT NOT NULL,
            library_location TEXT NOT NULL
        )
    """)

    conn.executemany(
        "INSERT INTO books (title, library_location) VALUES (?, ?)",
        books,
    )

print(f"Created {db_path} and added {len(books)} books.")

Created library.db and added 4 books.


In [25]:
class CompleteChat:

    def __init__(self, db_path="library.db"):
        self.db_path = db_path
        self.system_message = """
                                You are very helpful assistant that is able to answer to the user with concise and precise answer.
                                If you do not know, say so.
                                """
        self.tools = [
            {
                "type": "function", 
                "function": {
                                "name": "get_book_location",
                                "description": "Get the book library location",
                                "parameters": {
                                    "type": "object",
                                    "properties": {
                                        "title": {
                                            "type": "string",
                                            "description": "The book title",
                                        },
                                    },
                                    "required": ["title"],
                                    "additionalProperties": False
                                }
                            }
                }
        ]
        
    def models_selection(self, model_type):
        if model_type not in models_config:
            raise ValueError(f"We do not offer this particular model: {model_type}")
        return models_config.get(model_type)

    
    def get_book_location(self, title):
        with sqlite3.connect(self.db_path) as conn:
            cur = conn.cursor()
            cur.execute("SELECT title, library_location FROM books WHERE title = ?", (title,))

            result = cur.fetchone()
            if result:
                return f"{result[0]} is located at {result[1]}."
        return f"I couldn't find {title} in the library database."


    def handle_tool_call(self, tool_call):
        if tool_call.function.name == "get_book_location":
            arguments = json.loads(tool_call.function.arguments)
            return self.get_book_location(arguments['title'])
        return "This book is not available!"


    def chat(self, user_message, history, model_type):
        messages = [{"role": "system", "content": self.system_message}] + history
        messages.append({"role": "user", "content": user_message})

        response = openai.chat.completions.create(model=self.models_selection(model_type), messages=messages, tools=self.tools)

        assistant_message = response.choices[0].message
        if assistant_message.tool_calls:
            messages.append(assistant_message)

            tool_call = assistant_message.tool_calls[0]
            tool_result = self.handle_tool_call(tool_call)

            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": tool_result
            })

            stream = openai.chat.completions.create(model=self.models_selection(model_type), messages=messages, stream=True)

            answer = ""
            for chunk in stream:
                answer += chunk.choices[0].delta.content or ""

                yield answer
                
            self.history_message.append({"role": "user", "content": user_message})
            self.history_message.append({"role": "assistant", "content": answer})



In [26]:
chatbot = CompleteChat()

# # This is for single chat
# message_input = gr.Textbox(label="Your input", info="Check the location of library for book title", lines=5)
# model_selector = gr.Dropdown(["gpt", "google", "alibaba"], label="Select Model")
# message_output = gr.Markdown(label="Response:")

# gr.Interface(
#     fn=chatbot.chat,
#     inputs=[message_input, model_selector],
#     outputs=[message_output],
#     flagging_mode="never"
# ).launch()

# This is continuous chat

model_selector = gr.Dropdown(
    ["gpt", "google", "alibaba"],
    label="Select Model",
    value="gpt",
)

gr.ChatInterface(
    fn=chatbot.chat,
    additional_inputs=[model_selector],
    type="messages",
).launch()

* Running on local URL:  http://127.0.0.1:7870
* To create a public link, set `share=True` in `launch()`.
